In [ ]:
import os
import re
import json
import math
import torch
import gc
from datetime import datetime
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig
from google.colab import userdata

# HF_TOKEN = userdata.get("HF_TOKEN")
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

SYSTEM_PROMPT = (
    "..."
)

# print(f"GPU: {torch.cuda.get_device_name(0)}")
# print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── Answer extraction ──

def extract_boxed(text):
    """Extract content from the last \\boxed{...}, handling nested braces."""
    pass
    return 


def extract_ground_truth(raw_answer, gt_format):
    """Extract the ground-truth answer string from the dataset's answer field."""
    if gt_format == "hashmarks":
        m = re.search(r"####\s*(.+)", raw_answer)
        if m:
            return m.group(1).strip().replace(",", "")
        return raw_answer.strip()
    if gt_format == "boxed":
        boxed = extract_boxed(raw_answer)
        if boxed is not None:
            return boxed.strip()
        return raw_answer.strip()
    return raw_answer.strip()


def extract_model_answer(text):
    pass
    return 





# ── Model loading ──

def load_model(model_name=MODEL_NAME, lora_path=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )

    if lora_path:
        print(f"Loading LoRA adapter from {lora_path}")
        model = PeftModel.from_pretrained(model, lora_path)

    model.eval()
    tag = f" + LoRA({lora_path})" if lora_path else " (base)"
    print(f"Loaded: {model_name}{tag}")
    return model, tokenizer


def cleanup(*objects):
    """Delete model/tokenizer objects and free GPU memory."""
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


# ── Prompt building & batched generation ──

def build_prompts(tokenizer, questions):
    """Build chat-formatted prompt strings for a list of questions."""
    prompts = []
    for q in questions:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ]
        prompts.append(
            tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        )
    return prompts


def generate_batch(model, tokenizer, questions):
    """Generate responses for a batch of questions in one forward pass."""
    prompts = build_prompts(tokenizer, questions)
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=2048, do_sample=False)

    responses = []
    for i in range(len(questions)):
        new_tokens = out[i][prompt_len:]
        responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return responses


# ── Evaluation runner ──

def evaluate_gsm8k(model, tokenizer, num_samples=500, batch_size=16):
    """Zero-shot eval on GSM8K test set. Returns accuracy and records."""
    acc = 0
    records = None # all the answers to the test set
    return acc, records



In [ ]:
model, tokenizer = load_model()
base_acc, base_records = evaluate_gsm8k(model, tokenizer, num_samples=500)
print(f"\nBase model zero-shot accuracy: {base_acc:.2%}")
cleanup(model, tokenizer)   # important to cleanup so that the model is not loaded in memory

## SFT

In [ ]:
# ── Load model for training ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"  # right padding for training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

# ── LoRA config (attention-only, rank 8) ──
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Load & format GSM8K training data (3k) ──


# ── Training config ──
SFT_3K_DIR = "path"

training_args = SFTConfig(
    output_dir=SFT_3K_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    max_length=1024,
    completion_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data, # your training data
    processing_class=tokenizer,
)

print("\nStarting SFT 3k training...\n")
trainer.train()

# ── Save adapter ──
adapter_path = os.path.join(SFT_3K_DIR, "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"\nAdapter saved to {adapter_path}")

cleanup(model, tokenizer, trainer)

In [ ]:
model, tokenizer = load_model(lora_path="path")
sft3k_acc, sft3k_records = evaluate_gsm8k(model, tokenizer, num_samples=100)
print(f"\nSFT 3k zero-shot accuracy: {sft3k_acc:.2%}")
cleanup(model, tokenizer)